# EMVP-Protected ResNet — GPU Benchmark
**Plaintext vs EMVP (CPU) vs EMVP (GPU)** on MNIST and CIFAR-10.

Run top-to-bottom. Runtime → **T4 GPU** recommended (Runtime → Change runtime type).

In [ ]:
# ── 1. Clone repo and compile C extension ──────────────────────────────────
!git clone -b GPU-MPS https://github.com/lukemfitz/EMVPProject.git
%cd EMVPProject
!cc -O3 -shared -fPIC -o emvp_core.so emvp_core.c && echo 'C extension built OK'

In [ ]:
# ── 2. Check GPU ────────────────────────────────────────────────────────────
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"float64 : ", end="")
    try:
        a = torch.ones(2, 2, dtype=torch.float64, device='cuda')
        _ = a @ a
        print("supported")
    except Exception as e:
        print(f"NOT supported ({e})")
else:
    print("\nNo GPU detected — EMVP GPU path will fall back to C extension.")

In [ ]:
# ── 3. Imports and normalization constants ──────────────────────────────────
import sys, os, time, struct, gzip
import numpy as np
import torchvision, torchvision.transforms as T

sys.path.insert(0, '.')
from emvp_resnet import ResNet, softmax

_MNIST_MEAN = np.array([0.1307], dtype=np.float32)
_MNIST_STD  = np.array([0.3081], dtype=np.float32)
_CIFAR_MEAN = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32)
_CIFAR_STD  = np.array([0.2023, 0.1994, 0.2010], dtype=np.float32)

def norm_mnist(img): return (img - _MNIST_MEAN[:,None,None]) / _MNIST_STD[:,None,None]
def norm_cifar(img): return (img - _CIFAR_MEAN[:,None,None]) / _CIFAR_STD[:,None,None]

print('Imports OK')

In [ ]:
# ── 4. Load data ────────────────────────────────────────────────────────────
def load_dataset(name, n):
    cls = torchvision.datasets.MNIST if name == 'mnist' else torchvision.datasets.CIFAR10
    dset = cls('./data', train=False, download=True, transform=T.ToTensor())
    imgs, labs = [], []
    for img, label in dset:
        imgs.append(img.numpy())
        labs.append(label)
        if len(imgs) >= n: break
    return np.array(imgs, dtype=np.float32), np.array(labs, dtype=np.int64)

N_LOAD = 500
mnist_images,  mnist_labels  = load_dataset('mnist',   N_LOAD)
cifar_images,  cifar_labels  = load_dataset('cifar10', N_LOAD)
print(f'MNIST  : {mnist_images.shape}  labels {mnist_labels[:5]}')
print(f'CIFAR  : {cifar_images.shape}  labels {cifar_labels[:5]}')

In [ ]:
# ── 5. Benchmark runner ─────────────────────────────────────────────────────
def run_benchmark(dataset, raw_images, labels, normalize, weights,
                  C_in, n_blocks, n_plain=500, n_emvp=50,
                  k=16, s=4, seed=42):

    images = np.stack([normalize(raw_images[i]) for i in range(max(n_plain, n_emvp))])

    # Build CPU model (C extension, use_gpu=False)
    print(f'  Building models …', flush=True)
    model_cpu = ResNet(n_blocks=n_blocks, C_in=C_in, k=k, s=s, seed=seed,
                       weights=weights, use_gpu=False)
    # Build GPU model (same weights, use_gpu=True)
    model_gpu = ResNet(n_blocks=n_blocks, C_in=C_in, k=k, s=s, seed=seed,
                       weights=weights, use_gpu=True)
    print(f'  GPU device: {model_gpu.conv1.emvp.device}')

    # ── Plaintext ────────────────────────────────────────────────────────
    print(f'  Plaintext ({n_plain} images) …', flush=True)
    plain_preds, plain_logits = [], []
    t0 = time.perf_counter()
    for i in range(n_plain):
        lg = model_cpu.plaintext_forward(images[i])
        plain_preds.append(int(np.argmax(lg)))
        plain_logits.append(lg)
    t_plain = time.perf_counter() - t0
    acc_plain = np.mean(np.array(plain_preds) == labels[:n_plain])

    # ── EMVP CPU ─────────────────────────────────────────────────────────
    print(f'  EMVP CPU ({n_emvp} images) …', flush=True)
    cpu_preds, cpu_logits = [], []
    t0 = time.perf_counter()
    for i in range(n_emvp):
        lg = model_cpu.emvp_forward(images[i])
        cpu_preds.append(int(np.argmax(lg)))
        cpu_logits.append(lg)
        if (i + 1) % 10 == 0:
            print(f'    {i+1}/{n_emvp}  ({time.perf_counter()-t0:.1f}s)', flush=True)
    t_cpu = time.perf_counter() - t0
    acc_cpu = np.mean(np.array(cpu_preds) == labels[:n_emvp])

    # ── EMVP GPU ─────────────────────────────────────────────────────────
    print(f'  EMVP GPU ({n_emvp} images) …', flush=True)
    gpu_preds, gpu_logits = [], []
    t0 = time.perf_counter()
    for i in range(n_emvp):
        lg = model_gpu.emvp_forward(images[i])
        gpu_preds.append(int(np.argmax(lg)))
        gpu_logits.append(lg)
        if (i + 1) % 10 == 0:
            print(f'    {i+1}/{n_emvp}  ({time.perf_counter()-t0:.1f}s)', flush=True)
    t_gpu = time.perf_counter() - t0
    acc_gpu = np.mean(np.array(gpu_preds) == labels[:n_emvp])

    # ── Summary ───────────────────────────────────────────────────────────
    n_cmp = n_emvp
    agree_cpu = np.mean(np.array(plain_preds[:n_cmp]) == np.array(cpu_preds))
    agree_gpu = np.mean(np.array(plain_preds[:n_cmp]) == np.array(gpu_preds))
    l2_cpu = np.mean([np.linalg.norm(plain_logits[i]-cpu_logits[i]) for i in range(n_cmp)])
    l2_gpu = np.mean([np.linalg.norm(plain_logits[i]-gpu_logits[i]) for i in range(n_cmp)])

    ms_plain = t_plain / n_plain * 1000
    ms_cpu   = t_cpu   / n_emvp  * 1000
    ms_gpu   = t_gpu   / n_emvp  * 1000

    sep = '=' * 62
    print(f'\n{sep}')
    print(f'  {dataset.upper()}  Results')
    print(f'{sep}')
    print(f'  {"":30s}  {"Plaintext":>10}  {"EMVP CPU":>10}  {"EMVP GPU":>10}')
    print(f'  {"-"*30}  {"-"*10}  {"-"*10}  {"-"*10}')
    print(f'  {"Accuracy":30s}  {acc_plain*100:9.1f}%  {acc_cpu*100:9.1f}%  {acc_gpu*100:9.1f}%')
    print(f'  {"Pred agreement w/ plaintext":30s}  {"—":>10}  {agree_cpu*100:9.1f}%  {agree_gpu*100:9.1f}%')
    print(f'  {"Mean L2 logit error":30s}  {"—":>10}  {l2_cpu:10.4f}  {l2_gpu:10.4f}')
    print(f'  {"Speed (ms/image)":30s}  {ms_plain:10.2f}  {ms_cpu:10.1f}  {ms_gpu:10.1f}')
    print(f'  {"Slowdown vs plaintext":30s}  {"—":>10}  {ms_cpu/ms_plain:9.0f}×  {ms_gpu/ms_plain:9.0f}×')
    if ms_gpu > 0:
        print(f'  {"GPU speedup vs CPU":30s}  {"—":>10}  {"—":>10}  {ms_cpu/ms_gpu:9.1f}×')
    print(sep)

    return dict(
        acc_plain=acc_plain, acc_cpu=acc_cpu, acc_gpu=acc_gpu,
        ms_plain=ms_plain, ms_cpu=ms_cpu, ms_gpu=ms_gpu,
        agree_cpu=agree_cpu, agree_gpu=agree_gpu,
        l2_cpu=l2_cpu, l2_gpu=l2_gpu,
    )

print('Benchmark function defined.')

In [ ]:
# ── 6. MNIST Benchmark ──────────────────────────────────────────────────────
mnist_weights = np.load('mnist_weights.npy', allow_pickle=True).item()

print('MNIST Benchmark')
print('=' * 62)
mnist_results = run_benchmark(
    dataset='mnist',
    raw_images=mnist_images,
    labels=mnist_labels,
    normalize=norm_mnist,
    weights=mnist_weights,
    C_in=1, n_blocks=2,
    n_plain=500,
    n_emvp=50,
)

In [ ]:
# ── 7. CIFAR-10 Benchmark ───────────────────────────────────────────────────
cifar_weights = np.load('cifar10_weights.npy', allow_pickle=True).item()

print('CIFAR-10 Benchmark')
print('=' * 62)
cifar_results = run_benchmark(
    dataset='cifar10',
    raw_images=cifar_images,
    labels=cifar_labels,
    normalize=norm_cifar,
    weights=cifar_weights,
    C_in=3, n_blocks=2,
    n_plain=500,
    n_emvp=50,
)

In [ ]:
# ── 8. Combined summary table ───────────────────────────────────────────────
print()
print('╔══════════════════════════════════╦═══════════╦═══════════╗')
print('║ Metric                           ║   MNIST   ║  CIFAR-10 ║')
print('╠══════════════════════════════════╬═══════════╬═══════════╣')

rows = [
    ('Plaintext accuracy',        f"{mnist_results['acc_plain']*100:.1f}%",  f"{cifar_results['acc_plain']*100:.1f}%"),
    ('EMVP (CPU) accuracy',       f"{mnist_results['acc_cpu']*100:.1f}%",    f"{cifar_results['acc_cpu']*100:.1f}%"),
    ('EMVP (GPU) accuracy',       f"{mnist_results['acc_gpu']*100:.1f}%",    f"{cifar_results['acc_gpu']*100:.1f}%"),
    ('CPU pred agreement',        f"{mnist_results['agree_cpu']*100:.1f}%",  f"{cifar_results['agree_cpu']*100:.1f}%"),
    ('GPU pred agreement',        f"{mnist_results['agree_gpu']*100:.1f}%",  f"{cifar_results['agree_gpu']*100:.1f}%"),
    ('Mean L2 error (CPU)',       f"{mnist_results['l2_cpu']:.4f}",           f"{cifar_results['l2_cpu']:.4f}"),
    ('Mean L2 error (GPU)',       f"{mnist_results['l2_gpu']:.4f}",           f"{cifar_results['l2_gpu']:.4f}"),
    ('Plaintext speed (ms/img)',  f"{mnist_results['ms_plain']:.2f}",         f"{cifar_results['ms_plain']:.2f}"),
    ('EMVP CPU speed (ms/img)',   f"{mnist_results['ms_cpu']:.1f}",           f"{cifar_results['ms_cpu']:.1f}"),
    ('EMVP GPU speed (ms/img)',   f"{mnist_results['ms_gpu']:.1f}",           f"{cifar_results['ms_gpu']:.1f}"),
    ('GPU speedup vs CPU',        f"{mnist_results['ms_cpu']/mnist_results['ms_gpu']:.1f}×",
                                  f"{cifar_results['ms_cpu']/cifar_results['ms_gpu']:.1f}×"),
]

for label, m, c in rows:
    print(f'║ {label:<32} ║ {m:>9} ║ {c:>9} ║')

print('╚══════════════════════════════════╩═══════════╩═══════════╝')